## `a_paper_greek.ipynb` 26번 셀용 코드

`|r - sigma^2 / 2| <= 0.3`과 `sigma^2 <= T`를 동시에 만족하는 BS 파라미터 100개를 생성합니다.

In [ ]:
import numpy as np

from e_0_generate import generate_BS_params

MC_DT = 0.001


def generate_valid_bs_params(n_sets=100, seed=1234):
    """두 조건을 모두 만족하는 [r, sigma, T]를 정확히 n_sets개 생성합니다."""
    accepted_batches = []
    n_accepted = 0
    batch_index = 0

    while n_accepted < n_sets:
        batch_size = max(256, 2 * (n_sets - n_accepted))
        candidates = generate_BS_params(
            n_sets=batch_size,
            seed=seed if batch_index == 0 else None,
        )

        # 후속 MC 계산에서 사용하는 최종 T 값으로 먼저 맞춥니다.
        candidates[:, 2] = np.round(candidates[:, 2] / MC_DT) * MC_DT
        r = candidates[:, 0]
        sigma = candidates[:, 1]
        T = candidates[:, 2]

        valid = (np.abs(r - sigma**2 / 2) <= 0.3) & (sigma**2 <= T)
        accepted = candidates[valid]
        if len(accepted):
            accepted_batches.append(accepted)
            n_accepted += len(accepted)
        batch_index += 1

    return np.vstack(accepted_batches)[:n_sets]


# 다음 BS MAE 셀이 그대로 사용할 변수명입니다.
bs_para = generate_valid_bs_params(n_sets=100, seed=1234)

bs_r, bs_sigma, bs_T = bs_para.T
bs_abs_condition = np.abs(bs_r - bs_sigma**2 / 2) <= 0.3
bs_variance_condition = bs_sigma**2 <= bs_T

assert bs_para.shape == (100, 3)
assert np.all(bs_abs_condition & bs_variance_condition)
print(f"생성된 BS eta: {len(bs_para)}개")
print(f"|r - sigma^2 / 2| <= 0.3: {bs_abs_condition.sum()}/100")
print(f"sigma^2 <= T: {bs_variance_condition.sum()}/100")
bs_para


## `a_paper_greek.ipynb` 32번 셀용 코드

지정된 모든 파라미터 범위를 동시에 만족하는 Heston 파라미터 100개를 생성합니다.

In [ ]:
import numpy as np

from e_0_generate import generate_heston_params

MC_DT = 0.001
HESTON_COLUMNS = ["r", "kappa", "theta", "xi", "rho", "Y0", "T"]
HESTON_RANGES = {
    "r": (0.0025, 0.0975),
    "kappa": (1.3, 4.5),
    "theta": (0.035, 0.15),
    "xi": (0.15, 0.60),
    "rho": (-0.95, -0.05),
    "Y0": (0.005, 0.15),
    "T": (0.15, 2.85),
}


def generate_heston_params_in_range(n_sets=100, seed=1234):
    """모든 지정 범위를 동시에 만족하는 Heston 파라미터를 생성합니다."""
    accepted_batches = []
    n_accepted = 0
    batch_index = 0

    while n_accepted < n_sets:
        # 여러 범위를 동시에 통과해야 하므로 후보를 넉넉하게 생성합니다.
        batch_size = max(1000, 20 * (n_sets - n_accepted))
        candidates = generate_heston_params(
            n_sets=batch_size,
            seed=seed if batch_index == 0 else None,
        )
        candidates[:, 6] = np.round(candidates[:, 6] / MC_DT) * MC_DT

        valid = np.ones(batch_size, dtype=bool)
        for column_index, parameter in enumerate(HESTON_COLUMNS):
            lower, upper = HESTON_RANGES[parameter]
            valid &= candidates[:, column_index] >= lower
            valid &= candidates[:, column_index] <= upper

        accepted = candidates[valid]
        if len(accepted):
            accepted_batches.append(accepted)
            n_accepted += len(accepted)
        batch_index += 1

    return np.vstack(accepted_batches)[:n_sets]


# 다음 Heston MAE 셀이 그대로 사용할 변수명입니다.
hes_para = generate_heston_params_in_range(n_sets=100, seed=1234)

hes_range_checks = {}
for column_index, parameter in enumerate(HESTON_COLUMNS):
    lower, upper = HESTON_RANGES[parameter]
    values = hes_para[:, column_index]
    hes_range_checks[parameter] = bool(np.all((values >= lower) & (values <= upper)))

assert hes_para.shape == (100, 7)
assert all(hes_range_checks.values())
print(f"생성된 Heston eta: {len(hes_para)}개")
print("범위 검증:", hes_range_checks)
hes_para


# 파일의 x_T확인

In [2]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import h5py
import numpy as np
import pandas as pd


DEFAULT_BS_CHUNK_DIR = Path(r"E:\bs_chunks_correction") if os.name == "nt" else Path("/mnt/e/bs_chunks_correction")
DEFAULT_HESTON_CHUNK_DIR = Path(r"E:\heston_chunks_correction") if os.name == "nt" else Path("/mnt/e/heston_chunks_correction")
BS_CHUNK_DIR = Path(os.environ.get("BS_CHUNK_DIR", DEFAULT_BS_CHUNK_DIR))
HESTON_CHUNK_DIR = Path(os.environ.get("HESTON_CHUNK_DIR", DEFAULT_HESTON_CHUNK_DIR))
CHUNK_COUNT = int(os.environ.get("CVAE_CHUNK_COUNT", "100"))
MAX_WORKERS = int(os.environ.get("CVAE_CHUNK_WORKERS", str(min(10, os.cpu_count() or 1))))
ROW_BATCH_SIZE = int(os.environ.get("CVAE_CHUNK_ROW_BATCH_SIZE", "1000000"))

# 실행할 모델 선택: ("bs",), ("hes",), 또는 ("bs", "hes")
SCAN_MODELS = ("hes",) #("bs", "hes")
if os.environ.get("CVAE_SCAN_MODELS"):
    SCAN_MODELS = tuple(
        model.strip().lower()
        for model in os.environ["CVAE_SCAN_MODELS"].split(",")
        if model.strip()
    )
SCAN_MODELS = tuple(dict.fromkeys(SCAN_MODELS))
invalid_models = set(SCAN_MODELS) - {"bs", "hes"}
if not SCAN_MODELS or invalid_models:
    raise ValueError(
        f"SCAN_MODELS는 'bs', 'hes' 중 하나 이상이어야 합니다: {SCAN_MODELS}"
    )


def _expected_chunk_paths(model, chunk_dir, chunk_count):
    prefix = {"bs": "bs", "hes": "heston"}[model]
    paths = [chunk_dir / f"{prefix}_chunk_{index:03d}.h5" for index in range(chunk_count)]
    missing = [path for path in paths if not path.is_file()]
    if missing:
        preview = ", ".join(str(path) for path in missing[:5])
        suffix = f" 외 {len(missing) - 5}개" if len(missing) > 5 else ""
        raise FileNotFoundError(f"{model.upper()} chunk 파일 누락: {preview}{suffix}")
    return paths


def _scan_one_chunk(model, chunk_index, chunk_path, row_batch_size):
    with h5py.File(chunk_path, "r") as h5_file:
        if "paths" not in h5_file:
            raise KeyError(f"{chunk_path}: 'paths' 데이터셋이 없습니다.")

        paths = h5_file["paths"]
        if paths.ndim != 2 or paths.shape[1] < 2:
            raise ValueError(f"{chunk_path}: paths shape={paths.shape}, X_T 열(1)이 필요합니다.")

        n_rows = int(paths.shape[0])
        local_max = -np.inf
        local_row = -1
        nonfinite_count = 0

        for start in range(0, n_rows, row_batch_size):
            end = min(start + row_batch_size, n_rows)
            x_t = np.asarray(paths[start:end, 1])
            finite = np.isfinite(x_t)
            nonfinite_count += int((~finite).sum())
            if not finite.any():
                continue

            finite_values = np.where(finite, x_t, -np.inf)
            batch_position = int(np.argmax(finite_values))
            batch_max = float(finite_values[batch_position])
            if batch_max > local_max:
                local_max = batch_max
                local_row = start + batch_position

        if local_row < 0:
            local_max = np.nan
            eta_index = None
            result_row = None
        else:
            eta_index = int(paths[local_row, 0])
            result_row = local_row

    return {
        "model": model,
        "chunk_index": chunk_index,
        "chunk_file": str(chunk_path),
        "n_rows": n_rows,
        "max_x_t": local_max,
        "row_index": result_row,
        "eta_index": eta_index,
        "nonfinite_x_t_count": nonfinite_count,
    }


def scan_model_chunks_parallel(
    model, chunk_dir, chunk_count=100, max_workers=8, row_batch_size=1_000_000
):
    chunk_dir = Path(chunk_dir)
    chunk_paths = _expected_chunk_paths(model, chunk_dir, chunk_count)
    worker_count = max(1, min(max_workers, len(chunk_paths)))
    results = []

    with ThreadPoolExecutor(max_workers=worker_count) as executor:
        futures = {
            executor.submit(
                _scan_one_chunk, model, index, path, row_batch_size
            ): path
            for index, path in enumerate(chunk_paths)
        }
        for completed, future in enumerate(as_completed(futures), start=1):
            chunk_path = futures[future]
            try:
                results.append(future.result())
            except Exception as exc:
                raise RuntimeError(f"chunk 검사 실패: {chunk_path}") from exc

            print(f"[{model.upper()}] {completed}/{len(futures)}개 검사 완료")

    return pd.DataFrame(results).sort_values("chunk_index").reset_index(drop=True)


chunk_directories = {"bs": BS_CHUNK_DIR, "hes": HESTON_CHUNK_DIR}
selected_results = []
for model in SCAN_MODELS:
    selected_results.append(
        scan_model_chunks_parallel(
            model,
            chunk_directories[model],
            CHUNK_COUNT,
            MAX_WORKERS,
            ROW_BATCH_SIZE,
        )
    )
chunk_max_results = pd.concat(selected_results, ignore_index=True)

summary_rows = []
for model, model_results in chunk_max_results.groupby("model", sort=False):
    finite_results = model_results[np.isfinite(model_results["max_x_t"])]
    if finite_results.empty:
        raise ValueError(
            f"{model.upper()}: {len(model_results)}개 chunk 전체에 유한한 X_T 값이 없습니다."
        )
    maximum = finite_results.loc[finite_results["max_x_t"].idxmax()]
    summary_rows.append(
        {
            "model": model,
            "chunk_count": int(len(model_results)),
            "total_rows": int(model_results["n_rows"].sum()),
            "max_x_t": float(maximum["max_x_t"]),
            "chunk_index": int(maximum["chunk_index"]),
            "chunk_file": maximum["chunk_file"],
            "row_index": int(maximum["row_index"]),
            "eta_index": int(maximum["eta_index"]),
            "nonfinite_x_t_count": int(model_results["nonfinite_x_t_count"].sum()),
        }
    )

scan_summary = pd.DataFrame(summary_rows)
print("\n모델별 전체 최대 X_T")
print(scan_summary.to_string(index=False))
print("\n모든 chunk별 최대 X_T")
print(chunk_max_results.to_string(index=False))


[HES] 1/100개 검사 완료
[HES] 2/100개 검사 완료
[HES] 3/100개 검사 완료
[HES] 4/100개 검사 완료
[HES] 5/100개 검사 완료
[HES] 6/100개 검사 완료
[HES] 7/100개 검사 완료
[HES] 8/100개 검사 완료
[HES] 9/100개 검사 완료
[HES] 10/100개 검사 완료
[HES] 11/100개 검사 완료
[HES] 12/100개 검사 완료
[HES] 13/100개 검사 완료
[HES] 14/100개 검사 완료
[HES] 15/100개 검사 완료
[HES] 16/100개 검사 완료
[HES] 17/100개 검사 완료
[HES] 18/100개 검사 완료
[HES] 19/100개 검사 완료
[HES] 20/100개 검사 완료
[HES] 21/100개 검사 완료
[HES] 22/100개 검사 완료
[HES] 23/100개 검사 완료
[HES] 24/100개 검사 완료
[HES] 25/100개 검사 완료
[HES] 26/100개 검사 완료
[HES] 27/100개 검사 완료
[HES] 28/100개 검사 완료
[HES] 29/100개 검사 완료
[HES] 30/100개 검사 완료
[HES] 31/100개 검사 완료
[HES] 32/100개 검사 완료
[HES] 33/100개 검사 완료
[HES] 34/100개 검사 완료
[HES] 35/100개 검사 완료
[HES] 36/100개 검사 완료
[HES] 37/100개 검사 완료
[HES] 38/100개 검사 완료
[HES] 39/100개 검사 완료
[HES] 40/100개 검사 완료
[HES] 41/100개 검사 완료
[HES] 42/100개 검사 완료
[HES] 43/100개 검사 완료
[HES] 44/100개 검사 완료
[HES] 45/100개 검사 완료
[HES] 46/100개 검사 완료
[HES] 47/100개 검사 완료
[HES] 48/100개 검사 완료
[HES] 49/100개 검사 완료
[HES] 50/100개 검사 완료
[HES] 51/